## 💧Water Balance & Sustainability

This page analyzes groundwater sustainability using recharge, extraction, and utilization metrics.

##### Import Libraries
- pandas is used to load and handle dataset
- plotly.express is used to create interactive charts.

In [40]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


##### Load Dataset
- Reads the groundwater dataset into a DataFrame
- head() shows first 5 rows for quick preview

In [41]:
df = pd.read_csv("groundwater_ml_dataset_cleaned.csv")
df.head()

,state,district,annual_recharge,extractable_resource,annual_extraction,stage_of_development,category,extraction_ratio,utilization_rate,stress_level,risk_score,year
0,Himachal Pradesh,Himachal Pradesh,0.61,0.18,0.13,0.20,Safe,0.213115,0.722222,0.0020,0.374535,2024
1,Madhya Pradesh,Madhya Pradesh,27.00,1.68,0.17,7.04,Safe,0.006296,0.101190,0.0704,0.057075,2024
2,Andhra Pradesh,Alluri Sitharama Raju,43956.31,102516.48,2860.84,8890.60,Over Exploited,0.065084,0.027906,88.9060,17.818396,2024
3,Andhra Pradesh,Anakapalli,22443.69,38195.76,14423.44,6889.74,Over Exploited,0.642650,0.377619,68.8974,14.187588,2024
4,Andhra Pradesh,Ananthapuramu,40986.63,44150.14,1323.64,35512.65,Over Exploited,0.032294,0.029980,355.1265,71.050210,2024


### 🎨 Color Theme Used

White background with Plasma color scale:
- Light → Low values  
- Dark → High values  
- Clean, consistent and professional dashboard

### 1. Recharge vs Extraction
- Compares how much water is recharged vs extracted
- Helps identify over-exploited regions

In [42]:
fig1 = px.scatter(
    df,
    x='annual_recharge',
    y='annual_extraction',
    color='risk_score',
    color_continuous_scale=px.colors.sequential.Blugrn,
    title='Recharge vs Extraction'
)

fig1.update_layout(template='plotly_white', title_x=0.5)
fig1.show()

### 2. Extraction Ratio Distribution

Displays how extraction ratio varies across regions.

In [63]:

fig2 = px.box(
    df,
    x='category',
    y='extraction_ratio',
    title='Extraction Ratio Distribution',
    color='category',
    color_discrete_sequence=px.colors.sequential.Tealgrn
)

fig2.update_layout(
    template='plotly_white',
    title_x=0.5
)

fig2.show()

### 3 Utilization Rate Distribution

Shows how intensively groundwater is being used.

In [62]:

util_avg = df.groupby('category')['utilization_rate'].mean().reset_index()

fig3 = px.bar(
    util_avg,
    x='category',
    y='utilization_rate',
    title='Utilization Rate by Category',
    color='category',
    color_discrete_sequence=px.colors.sequential.Tealgrn
)

fig3.update_layout(
    template='plotly_white',
    title_x=0.5
)

fig3.show()

### 4 Water Surplus vs Deficit

Identifies regions with surplus or deficit groundwater.

In [61]:


df['water_balance'] = df['annual_recharge'] - df['annual_extraction']

df['balance_type'] = df['water_balance'].apply(
    lambda x: 'Surplus' if x > 0 else 'Deficit'
)

balance_count = df['balance_type'].value_counts().reset_index()
balance_count.columns = ['balance_type', 'count']

fig4 = px.bar(
    balance_count,
    x='balance_type',
    y='count',
    color='balance_type',
    title='Water Surplus vs Deficit',
    color_discrete_sequence=['#2CA58D', '#0B3C5D']  # green-blue
)

fig4.update_layout(
    template='plotly_white',
    title_x=0.5
)

fig4.show()

### 5 Sustainable vs Unsustainable Zones

In [60]:
df['sustainability'] = df['extraction_ratio'].apply(
    lambda x: 'Unsustainable' if x > 1 else 'Sustainable'
)

fig5 = px.pie(
    df,
    names='sustainability',
    title='Sustainable vs Unsustainable Zones',
    color_discrete_sequence=px.colors.sequential.Blugrn_r
)

fig5.update_layout(
    template='plotly_white',
    title_x=0.5
)

fig5.show()

### 6 Recharge Distribution by Category

Shows variation of recharge across different groundwater categories.

In [ ]:
fig6 = px.box(
    df,
    x='category',
    y='annual_recharge',
    title='Recharge per Category',
    color='category',
    color_discrete_sequence=px.colors.sequential.Tealgrn
)

fig6.update_layout(
    template='plotly_white',
    title_x=0.5
)

fig6.show()

### 7 Correlation Heatmap

Displays relationships between all numerical variables using annotated heatmap.

In [56]:
corr = df.corr(numeric_only=True)

fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale=px.colors.sequential.Tealgrn,
    title='Correlation Heatmap'
)

fig.update_layout(
    template='plotly_white',
    title_x=0.5
)

fig.show()

### 8. Water Balance → District-wise Interactive Bar Chart
- 💡 Insight:
Shows which districts are in surplus or deficit
Helps identify over-exploited districts
Dropdown se state-wise filtering possible

In [53]:
# -------------------------------
# STEP 1: Create Water Balance
# -------------------------------
df['water_balance'] = df['annual_recharge'] - df['annual_extraction']

df['balance_status'] = df['water_balance'].apply(
    lambda x: 'Surplus' if x >= 0 else 'Deficit'
)

# -------------------------------
# STEP 2: District-wise Aggregation
# -------------------------------
district_df = df.groupby(['district', 'balance_status']).size().unstack().fillna(0)

# -------------------------------
# STEP 3: Overall Data
# -------------------------------
overall = district_df.sum()

# -------------------------------
# STEP 4: State-wise Aggregation
# -------------------------------
state_group = df.groupby(['state', 'district', 'balance_status']).size().unstack().fillna(0)

# -------------------------------
# STEP 5: Create Figure
# -------------------------------
fig = go.Figure()

# Initial (Overall)
fig.add_trace(go.Bar(
    x=overall.index,
    y=overall.values,
    marker=dict(color=['#2CA58D', '#0B3C5D'])  # green-blue
))

# -------------------------------
# STEP 6: Dropdown (SLICER)
# -------------------------------
buttons = []

# Overall
buttons.append(dict(
    label="Overall",
    method="update",
    args=[{"x": [overall.index],
           "y": [overall.values]}]
))

# State filters
for state in df['state'].unique():
    state_data = state_group.loc[state].sum()

    buttons.append(dict(
        label=state,
        method="update",
        args=[{
            "x": [state_data.index],
            "y": [state_data.values]
        }]
    ))

# -------------------------------
# STEP 7: Layout
# -------------------------------
fig.update_layout(
    title="District-wise Water Balance (Surplus vs Deficit)",
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Water Balance Status",
    yaxis_title="District Count",
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        x=0,
        y=1.15,
        xanchor="left",
        yanchor="top"
    )]
)

# -------------------------------
# STEP 8: Show
# -------------------------------
fig.show()

### 9. Overuse Intensity Treemap (Top 10 Worst Districts)
- 💡 Insight:

Highlights most over-exploited districts based on extraction pressure.

In [50]:
df['overuse_intensity'] = df['annual_extraction'] - df['annual_recharge']

top_overuse = df.sort_values('overuse_intensity', ascending=False).head(10)

fig = px.treemap(
    top_overuse,
    path=['state', 'district'],
    values='overuse_intensity',
    color='overuse_intensity',
    color_continuous_scale='Tealgrn',
    title='Top 10 Overuse Districts'
)

fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

### 10. Interactive Water Stress Dashboard
-  Groundwater Stress Classification with Dynamic Filtering

This visualization represents the distribution of groundwater stress levels across regions and allows users to interactively filter data state-wise using a dropdown (slicer).

In [52]:
# -------------------------------
# STEP 1: Create Stress Category
# -------------------------------
df['stress_category'] = pd.cut(
    df['stress_level'],
    bins=[0, 0.3, 0.6, 1],
    labels=['Low Stress', 'Moderate Stress', 'High Stress']
)

# -------------------------------
# STEP 2: Overall Data
# -------------------------------
overall = df['stress_category'].value_counts().reindex(
    ['Low Stress','Moderate Stress','High Stress']
).fillna(0)

# -------------------------------
# STEP 3: State-wise Aggregation
# -------------------------------
state_group = df.groupby(['state','stress_category']).size().unstack().fillna(0)

# -------------------------------
# STEP 4: Color Palette (Green-Blue)
# -------------------------------
colors = ['#2CA58D', '#1F7A8C', '#0B3C5D']

# -------------------------------
# STEP 5: Create Donut Chart
# -------------------------------
fig = go.Figure()

fig.add_trace(go.Pie(
    labels=overall.index,
    values=overall.values,
    hole=0.55,
    marker=dict(colors=colors)
))

# -------------------------------
# STEP 6: Dropdown (ONLY Overall + States)
# -------------------------------
buttons = []

# Overall
buttons.append(dict(
    label="Overall",
    method="update",
    args=[{"values": [overall.values]}]
))

# Only State filters
for state in state_group.index:
    buttons.append(dict(
        label=state,
        method="update",
        args=[{"values": [state_group.loc[state].values]}]
    ))

# -------------------------------
# STEP 7: Layout (Top-Left)
# -------------------------------
fig.update_layout(
    title="Interactive Water Stress Dashboard",
    template="plotly_white",
    title_x=0.5,
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        showactive=True,
        x=0,
        y=1.15,
        xanchor="left",
        yanchor="top"
    )]
)

# -------------------------------
# STEP 8: Show
# -------------------------------
fig.show()

### 11. Water Balance Treemap
- Groundwater Surplus vs Deficit Structure

This treemap visualizes the hierarchical distribution of groundwater balance across states and districts. It helps in identifying regions with water surplus and deficit in a structured and visually intuitive way.

In [66]:
# Create water balance
df['water_balance'] = df['annual_recharge'] - df['annual_extraction']

df['balance_type'] = df['water_balance'].apply(
    lambda x: 'Surplus' if x > 0 else 'Deficit'
)

# Use absolute value for treemap size
df['balance_magnitude'] = df['water_balance'].abs()

# Aggregate
tree_df = df.groupby(['state', 'district', 'balance_type'])[
    'balance_magnitude'
].mean().reset_index()

# Treemap
fig = px.treemap(
    tree_df,
    path=['state', 'district', 'balance_type'],
    values='balance_magnitude',
    color='balance_magnitude',   # 🔥 apply gradient
    color_continuous_scale='Tealgrn',
    title='Water Balance Treemap (State → District)'
)

fig.update_layout(
    template='plotly_white',
    title_x=0.5
)

fig.show()